# Amharic Speech-to-Speech (S2S) Baseline Pipeline
## Research Latency Benchmarking on Google Colab

**Pipeline:** Audio → ASR (`snapwre/hohe-asr-amharic`) → LLM (`b1n1yam/gemma-2-27b-amharic-alpaca-sft`) → TTS (`gheero-Leyu/amharic-omnivoice-tts`).

This notebook is a reproducible **sequential baseline**. TTS latency is measured until the complete waveform is returned; true first-audio latency is not measured. LLM latency is measured to the first streamed text chunk. Real models are required: if a model cannot load, the benchmark stops rather than substituting a synthetic fallback.


---
# 1. Experiment Configuration

In this section, we define the global experiment configuration `CONFIG`. All execution hyperparameters, randomness seeds, and benchmark controls are centralized here to eliminate magic numbers.


In [1]:
import os, sys, random, numpy as np, torch

CONFIG = {
    "experiment_name": "amharic_s2s_baseline",
    "version": "2.0.0",
    "random_seed": 42,
    "n_runs": 5,
    "warmup_runs": 1,
    "dataset_pool_size": 80,
    "n_benchmark_samples": 12,
    "llm_max_new_tokens": 64,
    "llm_temperature": 0.3,
    "llm_top_p": 0.9,
    "llm_quantization": "4bit",
    "tts_num_steps": 32,
    "tts_sample_rate": 24000,
    "output_dir": "results",
    "audio_dir": "results/generated_audio",
    "figures_dir": "results/figures",
    "benchmark_audio_dir": "benchmark_audio",
}
for d in [CONFIG["output_dir"], CONFIG["audio_dir"], CONFIG["figures_dir"], CONFIG["benchmark_audio_dir"]]:
    os.makedirs(d, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(CONFIG["random_seed"])
print("✓ Baseline configuration initialized.")
print(f"Samples: {CONFIG['n_benchmark_samples']} | Runs/sample: {CONFIG['n_runs']} | TTS steps: {CONFIG['tts_num_steps']}")


✓ Baseline configuration initialized.
Samples: 12 | Runs/sample: 5 | TTS steps: 32


---
# 2. Environment / GPU Inspection

Before model execution, we inspect the hardware environment (GPU name, VRAM, system RAM, CUDA version, PyTorch, Transformers). This information is saved to `results/environment.json` to ensure reproducible experimental reporting.


In [2]:
import shutil
import psutil
import json

def inspect_environment():
    gpu_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_available else "None (CPU)"
    gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if gpu_available else 0.0

    vm = psutil.virtual_memory()
    disk = shutil.disk_usage("/")

    is_colab = "google.colab" in sys.modules or os.path.exists("/content")

    env_info = {
        "is_colab": is_colab,
        "python_version": sys.version.split()[0],
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda if gpu_available else "N/A",
        "gpu_available": gpu_available,
        "gpu_name": gpu_name,
        "gpu_vram_gb": round(gpu_vram_gb, 2),
        "system_ram_gb": round(vm.total / (1024**3), 2),
        "system_ram_available_gb": round(vm.available / (1024**3), 2),
        "disk_total_gb": round(disk.total / (1024**3), 2),
        "disk_free_gb": round(disk.free / (1024**3), 2),
    }

    print("=" * 50)
    print("COLAB ENVIRONMENT INSPECTION")
    print("=" * 50)
    print(f"Environment: {'Google Colab' if is_colab else 'Local / Custom Server'}")
    print(f"Python:      {env_info['python_version']}")
    print(f"PyTorch:     {env_info['pytorch_version']}")
    print(f"CUDA:        {env_info['cuda_version']}")
    print("-" * 50)
    print(f"GPU:         {env_info['gpu_name']}")
    print(f"GPU VRAM:    {env_info['gpu_vram_gb']} GB")
    print(f"System RAM:  {env_info['system_ram_gb']} GB (Available: {env_info['system_ram_available_gb']} GB)")
    print(f"Disk Free:   {env_info['disk_free_gb']} GB")
    print("=" * 50)

    env_file = os.path.join(CONFIG["output_dir"], "environment.json")
    with open(env_file, "w", encoding="utf-8") as f:
        json.dump(env_info, f, indent=2)
    print(f"✓ Saved environment info to {env_file}")
    return env_info

environment_info = inspect_environment()


COLAB ENVIRONMENT INSPECTION
Environment: Google Colab
Python:      3.13.15
PyTorch:     2.11.0+cu128
CUDA:        12.8
--------------------------------------------------
GPU:         Tesla T4
GPU VRAM:    14.56 GB
System RAM:  12.67 GB (Available: 8.81 GB)
Disk Free:   29.89 GB
✓ Saved environment info to results/environment.json


---
# 3. Dependency Installation & Environment Sanitization

We install the necessary audio and model execution libraries, cleanly uninstall any broken vision packages (since this is an audio-only pipeline), and verify all imported dependencies.


In [3]:
!pip uninstall -y torchvision -q
import sys
sys.modules["torchvision"] = None
!pip install -q "accelerate>=0.33.0" "bitsandbytes>=0.43.0" "peft>=0.12.0" soundfile "librosa>=0.10.0" scipy datasets psutil jiwer
!pip install -q --no-deps omnivoice || pip install -q --no-deps git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q -U git+https://github.com/huggingface/transformers.git

import transformers, soundfile as sf, librosa, pandas as pd, matplotlib.pyplot as plt
from jiwer import wer, cer

def verify_dependencies():
    required=["torch","torchaudio","transformers","accelerate","bitsandbytes","peft","soundfile","librosa","scipy","pandas","matplotlib","jiwer"]
    missing=[]
    for pkg in required:
        try:
            mod=__import__(pkg); print(f"[✓] {pkg:<15} {getattr(mod,'__version__','available')}")
        except Exception as e:
            missing.append(pkg); print(f"[✗] {pkg:<15} {e}")
    try:
        import omnivoice; print("[✓] omnivoice       available")
    except Exception as e:
        missing.append("omnivoice"); print(f"[✗] omnivoice       {e}")
    if missing: raise RuntimeError("Required dependencies missing: "+", ".join(missing))
verify_dependencies()


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[✓] torch           2.11.0+cu128
[✓] torchaudio      2.11.0+cu128
[✓] transformers    5.19.0.dev0
[✓] accelerate      1.14.0
[✓] bitsandbytes    0.50.2
[✓] peft            0.20.0
[✓] soundfile       0.14.0
[✓] librosa         0.11.0
[✓] scipy           1.16.3
[✓] pandas          2.2.3
[✓] matplotlib      3.10.0
[✓] jiwer           available
[✓] omnivoice       available


---
# 4. Model Card / Model Metadata

We record the documented metadata for each baseline model directly from their model cards.
*Note: Model-card metrics (e.g., WER/CER) serve as background context only and are never presented as experimental measurements from this benchmark.*


In [4]:
MODEL_CONFIG = {
    "asr": {
        "model_id": "snapwre/hohe-asr-amharic",
        "architecture": "Wav2Vec2ForCTC (CTC single-pass)",
        "parameter_count": "~317M",
        "training_data": "880 hours diverse Amharic audio (read, broadcast, calls, 5 regional dialects)",
        "documented_eval": "16.1% WER / 5.4% CER (model card claim; background only)",
        "framework": "transformers.AutoModelForCTC, Wav2Vec2Processor",
        "sample_rate": 16000,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "dtype": "float32",
        "limitations": "Reduced accuracy with overlapping speakers; no punctuation; numbers spelled phonetically"
    },
    "llm": {
        "model_id": "google/gemma-2-2b-it",
        "architecture": "Gemma-2 2B Instruction-tuned Causal LM",
        "parameter_count": "~2.6 Billion",
        "training_data": "Pre-trained on multilingual datasets + instruction tuning",
        "framework": "transformers.AutoModelForCausalLM, AutoTokenizer",
        "quantization": "None (native float16/bfloat16) or 8-bit",
        "device_map": "auto",
        "limitations": "Smaller general knowledge base but highly efficient; runs comfortably on T4 alongside audio pipelines"
    },
    "tts": {
        "model_id": "gheero-Leyu/amharic-omnivoice-tts",
        "architecture": "Non-autoregressive discrete diffusion language model",
        "parameter_count": "~612.6M",
        "training_data": "~331 hours Amharic audio (leyu-amharic-addis-ababa-dialect)",
        "documented_eval": "Zero-shot Amharic synthesis, 24kHz output, ~3GB VRAM on T4",
        "framework": "omnivoice.OmniVoice",
        "baseline_diffusion_steps": 32,
        "sample_rate": 24000,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "dtype": "float16",
        "limitations": "This baseline measures blocking complete-waveform generation; true first-audio latency is not exposed by this benchmark implementation"
    }
}

model_config_file = os.path.join(CONFIG["output_dir"], "model_config.json")
with open(model_config_file, "w", encoding="utf-8") as f:
    json.dump(MODEL_CONFIG, f, indent=2)
print(f"✓ Model metadata updated and saved to {model_config_file}")

✓ Model metadata updated and saved to results/model_config.json


---
# 5. Load ASR Model (`snapwre/hohe-asr-amharic`)

We load the CTC ASR model and processor, measure the cold-start loading time, and inspect GPU memory utilization.


In [5]:
import time
from transformers import AutoModelForCTC, Wav2Vec2Processor

def load_asr_model():
    model_id = MODEL_CONFIG["asr"]["model_id"]
    print(f"Loading ASR model: {model_id}...")
    start_time = time.perf_counter()

    device = torch.device(MODEL_CONFIG["asr"]["device"])

    # Load processor directly to bypass any multimodal vision hooks
    try:
        processor = Wav2Vec2Processor.from_pretrained(model_id)
    except Exception:
        from transformers import AutoProcessor
        processor = AutoProcessor.from_pretrained(model_id)

    model = AutoModelForCTC.from_pretrained(model_id).to(device)
    model.eval()

    load_time = time.perf_counter() - start_time
    vram_alloc = torch.cuda.memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0
    vram_res = torch.cuda.memory_reserved() / (1024**2) if torch.cuda.is_available() else 0.0

    print("-" * 40)
    print("ASR Model Loaded Successfully")
    print(f"Device:         {device}")
    print(f"Cold-load time: {load_time:.2f} s")
    print(f"VRAM Allocated: {vram_alloc:.1f} MB")
    print(f"VRAM Reserved:  {vram_res:.1f} MB")
    print("-" * 40)

    return {"processor": processor, "model": model, "device": device, "cold_load_time": load_time}

asr_bundle = load_asr_model()


Loading ASR model: snapwre/hohe-asr-amharic...


Loading weights:   0%|          | 0/797 [00:00<?, ?it/s]

----------------------------------------
ASR Model Loaded Successfully
Device:         cuda
Cold-load time: 11.33 s
VRAM Allocated: 2312.1 MB
VRAM Reserved:  2316.0 MB
----------------------------------------


In [6]:
import transformers.modeling_utils

# Bypass the buggy allocator warmup in transformers 5.x
if hasattr(transformers.modeling_utils, "caching_allocator_warmup"):
    transformers.modeling_utils.caching_allocator_warmup = lambda *args, **kwargs: None

---# 6. Load LLM (`google/gemma-4-E4B`)
The real 27B model is loaded with 4-bit NF4 quantization. If it cannot load on the available hardware, the notebook stops. No synthetic/proxy LLM is used.

In [7]:
from huggingface_hub import notebook_login

notebook_login()

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import transformers.modeling_utils, time, torch, os

if hasattr(transformers.modeling_utils, "caching_allocator_warmup"):
    transformers.modeling_utils.caching_allocator_warmup = lambda *args, **kwargs: None

def load_llm_model():
    model_id = MODEL_CONFIG["llm"]["model_id"]
    print(f"Loading REAL LLM: {model_id}...")
    start = time.perf_counter()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Using half-precision float16 directly since 2.6B fits easily without quantization
    if torch.cuda.is_available():
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.float16
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype=torch.float32,
            low_cpu_mem_usage=True
        )

    model.eval()
    load_time = time.perf_counter() - start
    print(f"✓ Real LLM loaded in {load_time:.2f}s")
    return {
        "tokenizer": tokenizer,
        "model": model,
        "device_map": getattr(model, "hf_device_map", "direct"),
        "cold_load_time": load_time
    }

llm_bundle = load_llm_model()

Loading REAL LLM: google/gemma-2-2b-it...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✓ Real LLM loaded in 161.41s


---
# 7. Load TTS Model (`gheero-Leyu/amharic-omnivoice-tts`)

The real OmniVoice model is loaded. If loading fails, the notebook stops rather than substituting a synthetic waveform. Baseline uses 32 diffusion steps.


In [9]:
def load_tts_model():
    from omnivoice import OmniVoice
    model_id=MODEL_CONFIG["tts"]["model_id"]
    print(f"Loading REAL TTS: {model_id}...")
    start=time.perf_counter()
    tts_model=OmniVoice.from_pretrained(
        model_id, device_map="cuda:0" if torch.cuda.is_available() else "cpu",
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
    load_time=time.perf_counter()-start
    print(f"✓ Real TTS loaded in {load_time:.2f}s")
    print(f"VRAM allocated: {torch.cuda.memory_allocated()/(1024**2):.1f} MB" if torch.cuda.is_available() else "CPU mode")
    return {"model":tts_model,"cold_load_time":load_time}
tts_bundle=load_tts_model()


Loading REAL TTS: gheero-Leyu/amharic-omnivoice-tts...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

✓ Real TTS loaded in 17.66s
VRAM allocated: 9238.9 MB


---
# 8. Audio Utilities

Audio loading, resampling, normalization, and sample synthesis utilities. Also includes Google Colab interactive file upload helper.


In [10]:
def load_audio(audio_input, target_sr=16000):
    """
    Loads and normalizes audio from filepath, raw array, or upload buffer.
    Returns: (audio_array_float32, sample_rate, duration_seconds, metadata_dict)
    """
    if isinstance(audio_input, str):
        wav, sr = librosa.load(audio_input, sr=target_sr, mono=True)
        file_size = os.path.getsize(audio_input) if os.path.exists(audio_input) else 0
        channels = 1
    elif isinstance(audio_input, tuple):
        wav, sr = audio_input
        if sr != target_sr:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
            sr = target_sr
        file_size = wav.nbytes
        channels = 1
    elif isinstance(audio_input, np.ndarray):
        wav = audio_input.astype(np.float32)
        sr = target_sr
        file_size = wav.nbytes
        channels = 1
    else:
        raise ValueError(f"Unsupported audio input type: {type(audio_input)}")

    # Peak normalization
    max_val = np.max(np.abs(wav))
    if max_val > 0:
        wav = wav / max_val

    duration = float(len(wav)) / float(sr)
    metadata = {
        "sample_rate": sr,
        "channels": channels,
        "duration": round(duration, 3),
        "file_size_bytes": file_size
    }
    return wav, sr, duration, metadata

def colab_upload_audio():
    """Interactive helper for Google Colab to upload custom audio files."""
    try:
        from google.colab import files
        print("Please upload your Amharic WAV audio file:")
        uploaded = files.upload()
        for filename in uploaded.keys():
            print(f"✓ Uploaded {filename}")
            return filename
    except Exception as e:
        print(f"Colab upload not active: {e}")
        return None


---
# 9. Timing Utilities

Precise latency instrumentation using `time.perf_counter()`. CUDA synchronization (`torch.cuda.synchronize()`) is enforced before and after GPU operations to prevent asynchronous kernel queueing from skewing latency measurements.


In [11]:
def cuda_sync():
    """Ensures all GPU asynchronous work is completed before timestamping."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()

class TimelineTracker:
    """
    Records high-resolution timestamps for pipeline events:
    - pipeline_start
    - audio_load_start / audio_load_end
    - speech_end
    - asr_start / asr_end
    - llm_request_start / llm_first_token / llm_end
    - tts_start / tts_first_audio / tts_end
    - pipeline_end
    """
    def __init__(self):
        self.events = {}

    def mark(self, event_name):
        cuda_sync()
        self.events[event_name] = time.perf_counter()

    def get_delta(self, start_event, end_event):
        t0 = self.events.get(start_event)
        t1 = self.events.get(end_event)
        if t0 is not None and t1 is not None:
            return round(t1 - t0, 4)
        return None

    def get_all_events(self):
        return self.events.copy()

print("✓ Timing and CUDA synchronization utilities initialized.")


✓ Timing and CUDA synchronization utilities initialized.


---
# 10. ASR Function (`transcribe`)

Measures ASR preprocessing, model inference, decoding, total latency and RTF. Also calculates reference-based WER/CER when the dataset provides a transcript.


In [12]:
def normalize_transcript(text):
    import unicodedata
    return " ".join(unicodedata.normalize("NFC", str(text or "")).strip().split())

def transcribe(audio_input, timeline=None, reference_text=None):
    if timeline: timeline.mark("audio_load_start")
    wav,sr,duration,meta=load_audio(audio_input,target_sr=16000)
    if timeline:
        timeline.mark("audio_load_end"); timeline.mark("speech_end"); timeline.mark("asr_preprocess_start")
    else:
        cuda_sync(); t0=time.perf_counter()
    processor=asr_bundle["processor"]; model=asr_bundle["model"]; device=asr_bundle["device"]
    inputs=processor(wav,sampling_rate=16000,return_tensors="pt").to(device)
    if timeline:
        timeline.mark("asr_preprocess_end"); timeline.mark("asr_inference_start")
    else:
        cuda_sync(); pre=time.perf_counter()-t0; ti=time.perf_counter()
    with torch.no_grad():
        # Use safe check to avoid converting multi-element tensor to boolean using the 'or' operator
        if "input_features" in inputs or hasattr(inputs, "input_features"):
            feat = getattr(inputs, "input_features", None)
            if feat is None:
                feat = inputs["input_features"]
            logits=model(feat).logits
        elif "input_values" in inputs or hasattr(inputs, "input_values"):
            val = getattr(inputs, "input_values", None)
            if val is None:
                val = inputs["input_values"]
            logits=model(val).logits
        else:
            logits=model(**inputs).logits
    if timeline:
        timeline.mark("asr_inference_end"); timeline.mark("asr_decode_start")
    else:
        cuda_sync(); inf=time.perf_counter()-ti; td=time.perf_counter()
    predicted_ids=torch.argmax(logits,dim=-1)
    transcription=processor.batch_decode(predicted_ids)[0].strip()
    if timeline:
        timeline.mark("asr_decode_end")
        pre=timeline.get_delta("asr_preprocess_start","asr_preprocess_end")
        inf=timeline.get_delta("asr_inference_start","asr_inference_end")
        dec=timeline.get_delta("asr_decode_start","asr_decode_end")
    else:
        cuda_sync(); dec=time.perf_counter()-td
    total=round(pre+inf+dec,4)
    rtf=round(total/duration,4) if duration>0 else 0
    ref=normalize_transcript(reference_text); hyp=normalize_transcript(transcription)
    w=wer(ref,hyp) if ref else np.nan
    c=cer(ref,hyp) if ref else np.nan
    return {"text":transcription,"reference_text":ref,"audio_duration":round(duration,3),
            "preprocessing_time":round(pre,4),"inference_time":round(inf,4),"decode_time":round(dec,4),
            "latency":total,"rtf":rtf,"wer":round(float(w),4) if not np.isnan(w) else np.nan,
            "cer":round(float(c),4) if not np.isnan(c) else np.nan,"metadata":meta}

---
# 11. LLM Function (`generate_response`)

Measures input/output tokens, **time to first streamed text**, total generation time, and tokens/sec. The first streamed text measurement is not described as a literal first-token latency.


In [13]:
from threading import Thread
from transformers import TextIteratorStreamer

AMHARIC_SYSTEM_PROMPT = "አንተ አጭር፣ ግልጽ እና ጠቃሚ ምላሽ በአማርኛ የምትሰጥ አጋዥ ረዳት ነህ። ምላሾችህ ቀጥተኛና አጭር ይሁኑ።"

def generate_response(transcript, conversation_history=None, timeline=None, max_new_tokens=64):
    tok=llm_bundle["tokenizer"]; model=llm_bundle["model"]
    messages=[{"role":"system","content":AMHARIC_SYSTEM_PROMPT},{"role":"user","content":transcript}]
    if getattr(tok,"chat_template",None):
        prompt=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    else:
        prompt=f"System: {AMHARIC_SYSTEM_PROMPT}\nUser: {transcript}\nAssistant:"
    device=getattr(model,"device",next(model.parameters()).device)
    inputs=tok(prompt,return_tensors="pt").to(device)
    streamer=TextIteratorStreamer(tok,skip_prompt=True,skip_special_tokens=True)
    kwargs=dict(**inputs,streamer=streamer,max_new_tokens=max_new_tokens,
                temperature=CONFIG["llm_temperature"],top_p=CONFIG["llm_top_p"],
                do_sample=True,pad_token_id=tok.pad_token_id)
    if timeline: timeline.mark("llm_request_start")
    else: cuda_sync(); t0=time.perf_counter()
    thread=Thread(target=model.generate,kwargs=kwargs); thread.start()
    text=""; first=None
    for chunk in streamer:
        if first is None:
            if timeline:
                timeline.mark("llm_first_text"); first=timeline.get_delta("llm_request_start","llm_first_text")
            else:
                cuda_sync(); first=round(time.perf_counter()-t0,4)
        text+=chunk
    thread.join()
    if timeline:
        timeline.mark("llm_end"); gen=timeline.get_delta("llm_request_start","llm_end")
    else:
        cuda_sync(); gen=round(time.perf_counter()-t0,4)
    out_tokens=len(tok.encode(text,add_special_tokens=False))
    return {"text":text.strip(),"input_tokens":int(inputs.input_ids.shape[1]),"output_tokens":out_tokens,
            "time_to_first_streamed_text":first,"generation_time":gen,
            "tokens_per_second":round(out_tokens/gen,2) if gen>0 else 0.0}


---
# 12. TTS Function (`synthesize`)

Measures complete waveform generation time and TTS RTF. No first-audio latency is reported because the current API returns the waveform only after generation completes.


In [14]:
def synthesize(text,num_steps=32,timeline=None):
    if timeline: timeline.mark("tts_start")
    else: cuda_sync(); t0=time.perf_counter()
    output=tts_bundle["model"].generate(text=text,num_steps=num_steps)
    if isinstance(output,list): wav=np.asarray(output[0],dtype=np.float32)
    elif isinstance(output,torch.Tensor): wav=output.squeeze().cpu().numpy().astype(np.float32)
    else: wav=np.asarray(output,dtype=np.float32)
    if timeline: timeline.mark("tts_end"); gen=timeline.get_delta("tts_start","tts_end")
    else: cuda_sync(); gen=round(time.perf_counter()-t0,4)
    sr=CONFIG["tts_sample_rate"]; dur=len(wav)/sr
    return {"audio":wav,"sample_rate":sr,"audio_duration":round(dur,3),
            "generation_time":gen,"rtf":round(gen/dur,4) if dur>0 else 0.0,"metadata":{"num_steps":num_steps}}


---
# 13. End-to-End Pipeline (`run_pipeline`)

The pipeline is sequential: Audio → ASR → LLM → TTS. The primary E2E metric is **speech-end → complete-audio latency**. This is not a true streaming first-audio measurement.


In [15]:
def run_pipeline(audio_input,reference_text=None,save_audio_path=None,num_tts_steps=None):
    steps=CONFIG["tts_num_steps"] if num_tts_steps is None else num_tts_steps
    timeline=TimelineTracker(); timeline.mark("pipeline_start")
    a=transcribe(audio_input,timeline=timeline,reference_text=reference_text)
    l=generate_response(a["text"],timeline=timeline,max_new_tokens=CONFIG["llm_max_new_tokens"])
    t=synthesize(l["text"],num_steps=steps,timeline=timeline)
    timeline.mark("pipeline_end")
    e2e=timeline.get_delta("speech_end","tts_end")
    total=timeline.get_delta("pipeline_start","pipeline_end")
    if save_audio_path: sf.write(save_audio_path,t["audio"],t["sample_rate"])
    return {"input":a["metadata"],
            "asr":{"transcript":a["text"],"reference_text":a["reference_text"],"duration":a["audio_duration"],
                   "preprocessing_time":a["preprocessing_time"],"inference_time":a["inference_time"],
                   "decode_time":a["decode_time"],"latency":a["latency"],"rtf":a["rtf"],"wer":a["wer"],"cer":a["cer"]},
            "llm":{"response":l["text"],"input_tokens":l["input_tokens"],"output_tokens":l["output_tokens"],
                   "time_to_first_streamed_text":l["time_to_first_streamed_text"],"generation_time":l["generation_time"],
                   "tokens_per_sec":l["tokens_per_second"]},
            "tts":{"duration":t["audio_duration"],"generation_time":t["generation_time"],"rtf":t["rtf"],"steps":steps},
            "end_to_end":{"e2e_completion_latency":e2e,"total_pipeline_latency":total},
            "audio_output":t["audio"],"sample_rate":t["sample_rate"]}


---
# 14. Single Smoke Test

We run a single verification sample through the complete pipeline, display the intermediate text representations, listen to the audio output, and print the latency breakdown table.


In [18]:
import IPython.display as ipd
import sys

# Redefine generate_response inline to fix the Gemma-2 system prompt role template issue.
def generate_response_fixed(transcript, conversation_history=None, timeline=None, max_new_tokens=64):
    tok = llm_bundle["tokenizer"]
    model = llm_bundle["model"]

    # Combine the system prompt into the first user message, as Gemma-2 template does not support a dedicated 'system' role.
    combined_user_content = f"{AMHARIC_SYSTEM_PROMPT}\n\nUser query: {transcript}"
    messages = [{"role": "user", "content": combined_user_content}]

    if getattr(tok, "chat_template", None):
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = f"User: {combined_user_content}\nAssistant:"

    device = getattr(model, "device", next(model.parameters()).device)
    inputs = tok(prompt, return_tensors="pt").to(device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(**inputs, streamer=streamer, max_new_tokens=max_new_tokens,
                temperature=CONFIG["llm_temperature"], top_p=CONFIG["llm_top_p"],
                do_sample=True, pad_token_id=tok.pad_token_id)

    if timeline:
        timeline.mark("llm_request_start")
    else:
        cuda_sync()
        t0 = time.perf_counter()

    thread = Thread(target=model.generate, kwargs=kwargs)
    thread.start()
    text = ""
    first = None
    for chunk in streamer:
        if first is None:
            if timeline:
                timeline.mark("llm_first_text")
                first = timeline.get_delta("llm_request_start", "llm_first_text")
            else:
                cuda_sync()
                first = round(time.perf_counter() - t0, 4)
        text += chunk
    thread.join()

    if timeline:
        timeline.mark("llm_end")
        gen = timeline.get_delta("llm_request_start", "llm_end")
    else:
        cuda_sync()
        gen = round(time.perf_counter() - t0, 4)

    out_tokens = len(tok.encode(text, add_special_tokens=False))
    return {
        "text": text.strip(),
        "input_tokens": int(inputs.input_ids.shape[1]),
        "output_tokens": out_tokens,
        "time_to_first_streamed_text": first,
        "generation_time": gen,
        "tokens_per_second": round(out_tokens / gen, 2) if gen > 0 else 0.0
    }

# Override global namespace function
sys.modules['__main__'].generate_response = generate_response_fixed

# Software smoke test only; this synthetic tone is never part of benchmark statistics.
sr_test = 16000
duration_test = 2.5
t = np.linspace(0, duration_test, int(sr_test * duration_test), endpoint=False)
smoke_wav = 0.5 * np.sin(2 * np.pi * 300 * t)
smoke_path = os.path.join(CONFIG["benchmark_audio_dir"], "smoke_test.wav")
sf.write(smoke_path, smoke_wav.astype(np.float32), sr_test)

print("Running software smoke test...")
smoke = run_pipeline(smoke_path)
print(f"ASR: '{smoke['asr']['transcript']}'")
print(f"LLM: '{smoke['llm']['response']}'")
print(f"ASR total: {smoke['asr']['latency']:.4f}s")
print(f"LLM first streamed text: {smoke['llm']['time_to_first_streamed_text']:.4f}s")
print(f"TTS complete waveform: {smoke['tts']['generation_time']:.4f}s")
print(f"E2E complete audio: {smoke['end_to_end']['e2e_completion_latency']:.4f}s")
ipd.Audio(smoke["audio_output"], rate=smoke["sample_rate"])

Running software smoke test...
ASR: '[AMH]'
LLM: 'እንደ አንድ አጋዥ ነው እንደሚገባ እንደ አንድ አጋዥ ነው እንደሚገባ እንደሚገባ እንደሚገባ እንደሚገባ'
ASR total: 0.3172s
LLM first streamed text: 3.1461s
TTS complete waveform: 4.8544s
E2E complete audio: 18.8192s


---
# 15. Benchmark Dataset

We select **12 real Amharic speech samples** from `gheero-Leyu/leyu-amharic-addis-ababa-dialect`. A larger candidate pool is collected first, then samples are selected using duration stratification. Available speaker/gender metadata and reference transcripts are retained.

The samples are **not** described as 12 categories.


In [ ]:
from datasets import load_dataset
print("Loading real Amharic speech...")
ds=load_dataset("gheero-Leyu/leyu-amharic-addis-ababa-dialect",split="train",streaming=True)

candidates=[]
for idx,item in enumerate(ds.take(CONFIG["dataset_pool_size"])):
    arr=np.asarray(item["audio"]["array"],dtype=np.float32); sr=int(item["audio"]["sampling_rate"])
    candidates.append({"source_index":idx,"audio_array":arr,"sampling_rate":sr,
                       "audio_duration":round(len(arr)/sr,3),"gender":item.get("gender"),
                       "speaker_id":item.get("speaker_id"),"reference_text":item.get("text","") or ""})
if len(candidates)<CONFIG["n_benchmark_samples"]:
    raise RuntimeError(f"Only {len(candidates)} candidates found.")

cdf=pd.DataFrame([{"i":x["source_index"],"duration":x["audio_duration"]} for x in candidates])
cdf["bin"]=pd.qcut(cdf["duration"],q=4,labels=False,duplicates="drop")
rng=np.random.default_rng(CONFIG["random_seed"])
selected=[]
groups=list(cdf.groupby("bin",dropna=False))
base=CONFIG["n_benchmark_samples"]//len(groups); rem=CONFIG["n_benchmark_samples"]%len(groups)
for j,(_,g) in enumerate(groups):
    n=min(base+(j<rem),len(g))
    if n: selected += rng.choice(g.index.to_numpy(),size=n,replace=False).tolist()
if len(selected)<CONFIG["n_benchmark_samples"]:
    remaining=[i for i in cdf.index if i not in selected]
    selected += remaining[:CONFIG["n_benchmark_samples"]-len(selected)]
selected=sorted(selected[:CONFIG["n_benchmark_samples"]])

BENCHMARK_SAMPLES=[]
for j,ci in enumerate(selected,1):
    x=candidates[int(ci)]; sid=f"sample_{j:02d}"
    path=os.path.join(CONFIG["benchmark_audio_dir"],sid+".wav")
    sf.write(path,x["audio_array"],x["sampling_rate"])
    BENCHMARK_SAMPLES.append({"id":sid,"source_index":x["source_index"],"gender":x["gender"],
                              "speaker_id":x["speaker_id"] or f"unknown_{j:02d}",
                              "reference_text":x["reference_text"],"path":path,"audio_duration":x["audio_duration"]})

manifest=pd.DataFrame([{"sample_id":x["id"],"source_index":x["source_index"],"speaker_id":x["speaker_id"],
                        "gender":x["gender"],"duration_sec":x["audio_duration"],
                        "reference_available":bool(str(x["reference_text"]).strip())} for x in BENCHMARK_SAMPLES])
manifest.to_csv(os.path.join(CONFIG["output_dir"],"benchmark_manifest.csv"),index=False)
print(f"✓ Selected {len(BENCHMARK_SAMPLES)} real speech samples.")
display(manifest)


Loading real Amharic speech...


README.md:   0%|          | 0.00/2.80k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

---
# 16. Baseline Benchmark Execution

Each selected sample is measured 5 times after one warm-up run. Warm-up is excluded from statistics. No optimization experiment is part of this baseline.


In [ ]:
import traceback
raw_runs_path=os.path.join(CONFIG["output_dir"],"raw_runs.jsonl")
print(f"Warm-up: {CONFIG['warmup_runs']} run")
for _ in range(CONFIG["warmup_runs"]):
    run_pipeline(BENCHMARK_SAMPLES[0]["path"],reference_text=BENCHMARK_SAMPLES[0]["reference_text"])
all_run_results=[]; failed_runs=[]
expected=len(BENCHMARK_SAMPLES)*CONFIG["n_runs"]; count=0
with open(raw_runs_path,"w",encoding="utf-8") as f:
    for sample in BENCHMARK_SAMPLES:
        for run_idx in range(1,CONFIG["n_runs"]+1):
            count+=1; print(f"[{count}/{expected}] {sample['id']} run {run_idx}...",end=" ",flush=True)
            try:
                out_path=os.path.join(CONFIG["audio_dir"],f"{sample['id']}_run{run_idx}.wav")
                r=run_pipeline(sample["path"],reference_text=sample["reference_text"],save_audio_path=out_path)
                rec={"sample_id":sample["id"],"source_index":sample["source_index"],"speaker_id":sample["speaker_id"],
                     "gender":sample["gender"],"run_id":run_idx,"input_audio_duration":r["asr"]["duration"],
                     "asr":r["asr"],"llm":r["llm"],"tts":r["tts"],"end_to_end":r["end_to_end"],
                     "output_audio_path":out_path,"status":"SUCCESS"}
                all_run_results.append(rec); f.write(json.dumps(rec,ensure_ascii=False)+"\n"); f.flush()
                print(f"OK | E2E complete {r['end_to_end']['e2e_completion_latency']:.3f}s")
            except Exception as e:
                rec={"sample_id":sample["id"],"run_id":run_idx,"error":str(e),"traceback":traceback.format_exc(),"status":"FAILED"}
                failed_runs.append(rec); f.write(json.dumps(rec,ensure_ascii=False)+"\n"); f.flush(); print(f"FAILED: {e}")
print(f"Finished: {len(all_run_results)} successes, {len(failed_runs)} failures.")


---
# 17. Metrics Calculation

Per-run and aggregate metrics include ASR preprocessing/inference/decoding/total latency, RTF, WER/CER; LLM first streamed text, generation time and tokens/sec; TTS generation/RTF; and end-to-end speech-end → complete-audio latency. Statistics: N, mean, median, std, min, max, P90.


In [ ]:
def compute_metrics(runs):
    if not runs: raise RuntimeError("No successful runs.")
    rows=[]
    for r in runs:
        rows.append({
            "sample_id":r["sample_id"],"speaker_id":r["speaker_id"],"gender":r["gender"],
            "input_audio_duration":r["input_audio_duration"],
            "asr_preprocessing_time":r["asr"]["preprocessing_time"],"asr_inference_time":r["asr"]["inference_time"],
            "asr_decode_time":r["asr"]["decode_time"],"asr_latency":r["asr"]["latency"],"asr_rtf":r["asr"]["rtf"],
            "asr_wer":r["asr"]["wer"],"asr_cer":r["asr"]["cer"],
            "llm_input_tokens":r["llm"]["input_tokens"],"llm_output_tokens":r["llm"]["output_tokens"],
            "llm_first_streamed_text":r["llm"]["time_to_first_streamed_text"],"llm_generation_time":r["llm"]["generation_time"],
            "llm_tokens_per_sec":r["llm"]["tokens_per_sec"],
            "tts_generation_time":r["tts"]["generation_time"],"tts_audio_duration":r["tts"]["duration"],"tts_rtf":r["tts"]["rtf"],
            "e2e_completion_latency":r["end_to_end"]["e2e_completion_latency"],"total_pipeline_latency":r["end_to_end"]["total_pipeline_latency"]})
    df=pd.DataFrame(rows)
    cols=[c for c in df.columns if c not in ("sample_id","speaker_id","gender")]
    stats={}
    for c in cols:
        s=pd.to_numeric(df[c],errors="coerce").dropna()
        if len(s): stats[c]={"n":int(len(s)),"mean":round(float(s.mean()),4),"median":round(float(s.median()),4),
                              "std":round(float(s.std()),4) if len(s)>1 else 0.0,"min":round(float(s.min()),4),
                              "max":round(float(s.max()),4),"p90":round(float(s.quantile(.9)),4)}
    with open(os.path.join(CONFIG["output_dir"],"summary.json"),"w") as f: json.dump(stats,f,indent=2)
    df.to_csv(os.path.join(CONFIG["output_dir"],"per_run_metrics.csv"),index=False)
    return df,stats
df_runs,summary_metrics=compute_metrics(all_run_results)
display(pd.DataFrame(summary_metrics).T[["n","mean","median","std","min","max","p90"]])


---
# 18. Visualizations

Figures: median stage latency, E2E completion-latency distribution, audio duration vs ASR latency, and ASR preprocessing/inference/decoding breakdown.


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,10))
axes[0,0].bar(["ASR","LLM","TTS"],[summary_metrics["asr_latency"]["median"],summary_metrics["llm_generation_time"]["median"],summary_metrics["tts_generation_time"]["median"]])
axes[0,0].set_title("Median Stage Latency"); axes[0,0].set_ylabel("Seconds")
axes[0,1].hist(df_runs["e2e_completion_latency"].dropna(),bins=10)
axes[0,1].axvline(summary_metrics["e2e_completion_latency"]["median"],linestyle="--",label="Median")
axes[0,1].set_title("E2E Speech-End → Complete-Audio Latency"); axes[0,1].set_xlabel("Seconds"); axes[0,1].legend()
axes[1,0].scatter(df_runs["input_audio_duration"],df_runs["asr_latency"],alpha=.7)
if df_runs["input_audio_duration"].nunique()>1:
    m,b=np.polyfit(df_runs["input_audio_duration"],df_runs["asr_latency"],1)
    x=np.linspace(df_runs["input_audio_duration"].min(),df_runs["input_audio_duration"].max(),50)
    axes[1,0].plot(x,m*x+b,linestyle="--")
axes[1,0].set_title("Audio Duration vs ASR Latency"); axes[1,0].set_xlabel("Audio Duration (s)"); axes[1,0].set_ylabel("ASR Latency (s)")
axes[1,1].bar(["Preprocess","Inference","Decode"],[summary_metrics["asr_preprocessing_time"]["median"],summary_metrics["asr_inference_time"]["median"],summary_metrics["asr_decode_time"]["median"]])
axes[1,1].set_title("ASR Latency Breakdown"); axes[1,1].set_ylabel("Seconds")
plt.tight_layout()
fig_path=os.path.join(CONFIG["figures_dir"],"baseline_latency_analysis.png"); plt.savefig(fig_path,dpi=300); plt.show()
print(f"✓ Saved {fig_path}")


---
# 19. Bottleneck Analysis

Identifies the largest measured median stage and reports the audio-duration/ASR-latency relationship. It does not prescribe an optimization.


In [ ]:
stage_medians={"ASR":summary_metrics["asr_latency"]["median"],"LLM generation":summary_metrics["llm_generation_time"]["median"],"TTS":summary_metrics["tts_generation_time"]["median"]}
dominant_stage=max(stage_medians,key=stage_medians.get)
sum_stage=sum(stage_medians.values())
bottleneck_percentage=stage_medians[dominant_stage]/sum_stage*100 if sum_stage else np.nan
duration_asr_corr=df_runs[["input_audio_duration","asr_latency"]].corr().iloc[0,1] if df_runs["input_audio_duration"].nunique()>1 else np.nan
print("="*60)
print(f"Median E2E completion: {summary_metrics['e2e_completion_latency']['median']:.4f}s")
print(f"Measured dominant stage: {dominant_stage}")
print(f"Share of summed stage medians: {bottleneck_percentage:.1f}%")
print(f"Median ASR RTF: {summary_metrics['asr_rtf']['median']:.4f}")
print(f"Median LLM first streamed text: {summary_metrics['llm_first_streamed_text']['median']:.4f}s")
print(f"Median TTS generation: {summary_metrics['tts_generation_time']['median']:.4f}s")
print(f"Audio-duration/ASR correlation: {duration_asr_corr:.4f}")
print("="*60)
print("Optimization choice is intentionally deferred until this baseline is reviewed.")


---
# 20. Save Results & Generate Research Report

The report summarizes measured baseline latency, ASR quality sanity metrics, hardware, dataset scope, and limitations. It does not include optimization results.


In [ ]:
def fmt(k):
    s=summary_metrics.get(k)
    return f"{s['median']:.4f}s median (p90 {s['p90']:.4f}s)" if s else "N/A"

report=f"""# Amharic Speech-to-Speech Baseline Research Report

## 1. Executive Summary
Sequential baseline using:
- ASR: `{MODEL_CONFIG['asr']['model_id']}`
- LLM: `{MODEL_CONFIG['llm']['model_id']}` (4-bit)
- TTS: `{MODEL_CONFIG['tts']['model_id']}` ({CONFIG['tts_num_steps']} diffusion steps)

Dataset: {len(BENCHMARK_SAMPLES)} real Amharic speech samples selected from a larger candidate pool using duration stratification. Each sample was measured {CONFIG['n_runs']} times after warm-up.

This benchmark is sequential. TTS latency means speech-end to complete waveform availability; true first-audio latency is not measured.

## 2. Environment
- GPU: {environment_info['gpu_name']} ({environment_info['gpu_vram_gb']} GB VRAM)
- RAM: {environment_info['system_ram_gb']} GB
- PyTorch: {environment_info['pytorch_version']}
- CUDA: {environment_info['cuda_version']}
- Python: {environment_info['python_version']}

## 3. Results
| Metric | Result |
|---|---:|
| ASR preprocessing | {fmt('asr_preprocessing_time')} |
| ASR inference | {fmt('asr_inference_time')} |
| ASR decoding | {fmt('asr_decode_time')} |
| ASR total | {fmt('asr_latency')} |
| ASR RTF | {fmt('asr_rtf')} |
| ASR WER | {fmt('asr_wer')} |
| ASR CER | {fmt('asr_cer')} |
| LLM first streamed text | {fmt('llm_first_streamed_text')} |
| LLM generation | {fmt('llm_generation_time')} |
| LLM tokens/sec | {fmt('llm_tokens_per_sec')} |
| TTS generation | {fmt('tts_generation_time')} |
| TTS RTF | {fmt('tts_rtf')} |
| E2E speech-end → complete audio | {fmt('e2e_completion_latency')} |

## 4. Bottleneck
The largest measured median stage is **{dominant_stage}**, accounting for approximately {bottleneck_percentage:.1f}% of the sum of median ASR, LLM-generation, and TTS stage latencies. This is a descriptive baseline finding, not an optimization conclusion.

## 5. ASR Quality
WER/CER are reference-based sanity metrics using the dataset transcripts after minimal Unicode NFC and whitespace normalization. They are not presented as a formal held-out model evaluation.

## 6. Limitations
- Only {len(BENCHMARK_SAMPLES)} speech samples were benchmarked.
- Repeated runs are not independent speakers.
- The pipeline is sequential, not true streaming.
- TTS first-audio latency is unavailable from the current blocking generation API.
- WER/CER depend on the supplied reference transcripts.
- Hardware, quantization, offloading, and software versions affect latency.

## 7. Next Step
Use this frozen baseline to choose one targeted optimization experiment based on the measured bottleneck. Optimization is intentionally separated from this notebook.
"""
report_path=os.path.join(CONFIG["output_dir"],"baseline_report.md")
with open(report_path,"w",encoding="utf-8") as f: f.write(report)
print(f"✓ Research report saved to {report_path}")


---
# 21. Final Baseline Summary Banner

This is the end of the baseline notebook. Optimization should be run separately so the baseline measurements remain frozen and auditable.


In [ ]:
print(f"""
==================================================
AMHARIC SPEECH-TO-SPEECH BASELINE
==================================================
ASR
  Model: {MODEL_CONFIG['asr']['model_id']}
  Median latency: {summary_metrics['asr_latency']['median']:.4f}s
  Median inference: {summary_metrics['asr_inference_time']['median']:.4f}s
  Median WER: {summary_metrics.get('asr_wer',{}).get('median',float('nan')):.4f}
  Median CER: {summary_metrics.get('asr_cer',{}).get('median',float('nan')):.4f}
  RTF: {summary_metrics['asr_rtf']['median']:.4f}

LLM
  Model: {MODEL_CONFIG['llm']['model_id']}
  First streamed text: {summary_metrics['llm_first_streamed_text']['median']:.4f}s
  Median generation: {summary_metrics['llm_generation_time']['median']:.4f}s
  Tokens/sec: {summary_metrics['llm_tokens_per_sec']['median']:.2f}

TTS
  Model: {MODEL_CONFIG['tts']['model_id']}
  Steps: {CONFIG['tts_num_steps']}
  Median generation: {summary_metrics['tts_generation_time']['median']:.4f}s
  RTF: {summary_metrics['tts_rtf']['median']:.4f}

END-TO-END
  Speech → complete audio: {summary_metrics['e2e_completion_latency']['median']:.4f}s

MEASURED BOTTLENECK
  {dominant_stage} ({bottleneck_percentage:.1f}% of summed stage medians)

No first-audio claim. No optimization included.
==================================================
""")
